# Sequential Training

We train the models sequentially, across multiple days.

- **Dataset:** TOTF.PA (Euronext Paris), 25 daily LOB snapshots
- **Scaler:** Quantile (Box-Cox + z-score)
- **Models:**
    - Transformer + OC-SVM (hybrid)
    - PNN (Probabilistic Neural Network)
    - PRAE (Probabilistic Robust Autoencoder)
- **Training Strategy:** For each of the first 24 days, use the first hour of market data split into alternating 5-minute blocks of training and validation.
- **Test Day:** Day 25 (held out entirely).
- **Features:** base, tao (weighted imbalance), poutre (rapidity / event flow), hawkes (memory), ofi (order flow imbalance).

In [1]:
import os
import sys
import glob
import logging
import copy

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
import joblib

sys.path.insert(0, os.path.abspath(".."))

from detection.data.datasets import get_lob, IndexDataset
from detection.data.preprocessing import clean_lob, filter_market_hours
from detection.data.loaders import create_sequences
from detection.data.scalers import scaler as QuantileScaler
from detection.features.dynamics import compute_dynamics, compute_elasticity
from detection.features.event_flow import compute_event_flow
from detection.features.hawkes import compute_hawkes
from detection.features.imbalance import compute_imbalance, compute_weighted_imbalance
from detection.features.ofi import compute_ofi
from detection.features.volatility import compute_volatility
from detection.models.transformer import BottleneckTransformer
from detection.models.hybrid import TransformerOCSVM
from detection.models.pnn import PNN
from detection.models.prae import PRAE, calculate_heuristic_lambda
from detection.trainers.training import Trainer
from detection.trainers.callbacks import EarlyStopping

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("training")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", DEVICE)

2026-02-25 00:10:44 [INFO] training: Device: cuda


## Configuration

In [2]:
# Paths
DATA_DIR = os.path.join("..", "data", "TOTF.PA-book")
RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Data parameters
TIME_COL = "xltime"
MARKET_OPEN_HOUR = 9.0   # Euronext Paris continuous session
MARKET_CLOSE_HOUR = 17.5
FIRST_HOUR_MINUTES = 60
TRAIN_BLOCK_MINUTES = 5
VAL_BLOCK_MINUTES = 5

# Feature engineering
FEATURE_SETS = ["base", "tao", "poutre", "hawkes", "ofi"]
WARMUP_STEPS = 3000
WINDOW = 50

# Preprocessing
SEQ_LENGTH = 25
BATCH_SIZE = 64
TARGET_COL = "log_return"

# Training
EPOCHS = 1000
LR = 1e-3
PATIENCE = 15

# Model architectures
TRANSFORMER_CFG = dict(model_dim=64, num_heads=4, num_layers=2, representation_dim=128)
PNN_HIDDEN_DIM = 64
PRAE_SIGMA = 0.5

# OC-SVM (Nyström approximation + linear SGD on CUDA)
OCSVM_NU = 0.01
NYSTROEM_COMPONENTS = 300
OCSVM_SGD_LR = 0.01
OCSVM_SGD_EPOCHS = 500

# File listing
FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv.gz")))
# NUM_TRAIN_DAYS = len(FILES) - 1  # Last day is held out for testing
NUM_TRAIN_DAYS = 3
logger.info("Found %d daily files.  Training on %d days, testing on day %d.",
            len(FILES), NUM_TRAIN_DAYS, len(FILES))

2026-02-25 00:10:44 [INFO] training: Found 25 daily files.  Training on 3 days, testing on day 25.


## Helper functions

Reusable helpers for:

1. **Feature engineering** -- wraps all detection.features modules.
2. **Scaling and sequencing** -- fits a scaler on training data and builds 3-D sequences.
3. **Time-block splitting** -- splits the first hour into alternating 5-min train/val blocks.
4. **DataLoader construction** -- builds loaders per model type.

In [3]:
def load_and_clean(filepath):
    """Load a single daily file, clean artifacts, filter to market hours."""
    df = get_lob(filepath)
    df = clean_lob(df)
    df = filter_market_hours(df, time_col=TIME_COL,
                             market_open_hour=MARKET_OPEN_HOUR,
                             market_close_hour=MARKET_CLOSE_HOUR)
    return df


def engineer_features(df):
    """Compute all feature sets from raw LOB data, return cleaned DataFrame."""
    features = pd.DataFrame(index=df.index)

    # base
    df = compute_imbalance(df)
    features["L1_Imbalance"] = df["L1_Imbalance"]
    features["L5_Imbalance"] = df["L5_Imbalance"]
    features = compute_dynamics(df, features, window=WINDOW)
    features = compute_elasticity(df, features)
    features = compute_volatility(df, features, window=WINDOW)

    # tao - weighted imbalance
    features["Weighted_Imbalance_decreasing"] = compute_weighted_imbalance(
        df, weights=[0.1, 0.1, 0.2, 0.2, 0.4], levels=5)
    features["Weighted_Imbalance_increasing"] = compute_weighted_imbalance(
        df, weights=[0.4, 0.2, 0.2, 0.1, 0.1], levels=5)
    features["Weighted_Imbalance_constant"] = compute_weighted_imbalance(
        df, weights=[0.2, 0.2, 0.2, 0.2, 0.2], levels=5)

    # poutre - event flow / rapidity
    features = compute_event_flow(df, features)

    # hawkes - memory
    features = compute_hawkes(df, features)

    # ofi - order flow imbalance
    features = compute_ofi(df, features)

    # cleanup
    features.replace([np.inf, -np.inf], np.nan, inplace=True)
    features = features.fillna(0)

    if len(features) > WARMUP_STEPS:
        features = features.iloc[WARMUP_STEPS:].reset_index(drop=True)

    lower = features.quantile(0.001)
    upper = features.quantile(0.999)
    features = features.clip(lower=lower, upper=upper, axis=1)

    # drop constant columns
    std_devs = features.std()
    drop_cols = std_devs[std_devs < 1e-9].index.tolist()
    if drop_cols:
        features = features.drop(columns=drop_cols)

    return features


def split_first_hour_blocks(df, features):
    """Split the first hour of a day into 5-min train/val blocks.

    Returns
    -------
    train_features, val_features : DataFrames
    """
    time_factor = 1.0 / (24.0 * 60.0)  # 1 minute as a fraction of a day
    base_date = np.floor(df[TIME_COL].iloc[0])
    start_time = base_date + MARKET_OPEN_HOUR / 24.0

    # The warmup trim may have removed early rows, so align using features index
    # We use the raw df xltime values aligned to feature rows after warmup
    if len(df) > len(features):
        df_aligned = df.iloc[-len(features):].reset_index(drop=True)
    else:
        df_aligned = df.reset_index(drop=True)

    xltime = df_aligned[TIME_COL].values
    train_mask = np.zeros(len(features), dtype=bool)
    val_mask = np.zeros(len(features), dtype=bool)
    block_duration = (TRAIN_BLOCK_MINUTES + VAL_BLOCK_MINUTES) * time_factor
    num_blocks = int(FIRST_HOUR_MINUTES / (TRAIN_BLOCK_MINUTES + VAL_BLOCK_MINUTES))

    for b in range(num_blocks):
        block_start = start_time + b * block_duration
        train_end = block_start + TRAIN_BLOCK_MINUTES * time_factor
        val_end = train_end + VAL_BLOCK_MINUTES * time_factor
        train_mask |= (xltime >= block_start) & (xltime < train_end)
        val_mask |= (xltime >= train_end) & (xltime < val_end)

    train_features = features.loc[train_mask].reset_index(drop=True)
    val_features = features.loc[val_mask].reset_index(drop=True)
    return train_features, val_features


def scale_and_create_loaders(train_feat, val_feat, scaler, model_type, feature_names, fit_scaler=True):
    """Scale features, create sequences, and build DataLoaders.

    Parameters
    ----------
    fit_scaler : bool
        If True, fit the scaler on training data.  If False, use transform only (for incremental days).

    Returns
    -------
    train_loader, val_loader, scaler, feature_names
    """
    if fit_scaler:
        train_scaled = scaler.fit_transform(train_feat.values.astype(np.float32)).astype(np.float32)
    else:
        train_scaled = scaler.transform(train_feat.values.astype(np.float32)).astype(np.float32)

    val_scaled = scaler.transform(val_feat.values.astype(np.float32)).astype(np.float32)

    train_seqs = create_sequences(train_scaled, SEQ_LENGTH)
    val_seqs = create_sequences(val_scaled, SEQ_LENGTH)

    if len(train_seqs) == 0 or len(val_seqs) == 0:
        return None, None, scaler, feature_names

    target_idx = feature_names.index(TARGET_COL)
    train_targets = train_scaled[SEQ_LENGTH:, target_idx][:len(train_seqs)]
    val_targets = val_scaled[SEQ_LENGTH:, target_idx][:len(val_seqs)]

    x_train = torch.tensor(train_seqs, dtype=torch.float32)
    x_val = torch.tensor(val_seqs, dtype=torch.float32)

    if model_type == "pnn":
        y_train = torch.tensor(train_targets, dtype=torch.float32).unsqueeze(1)
        y_val = torch.tensor(val_targets, dtype=torch.float32).unsqueeze(1)
        train_ds = TensorDataset(x_train.reshape(x_train.size(0), -1), y_train)
        val_ds = TensorDataset(x_val.reshape(x_val.size(0), -1), y_val)
    elif model_type == "prae":
        train_ds = IndexDataset(TensorDataset(x_train, x_train))
        val_ds = IndexDataset(TensorDataset(x_val, x_val))
    else:  # transformer_ocsvm
        train_ds = TensorDataset(x_train, x_train)
        val_ds = TensorDataset(x_val, x_val)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, scaler, feature_names

In [4]:
def build_fresh_model(model_type, num_features):
    """Factory: create a fresh (untrained) model instance."""
    if model_type == "transformer_ocsvm":
        transformer = BottleneckTransformer(
            num_features=num_features, sequence_length=SEQ_LENGTH, **TRANSFORMER_CFG)
        early_stop = EarlyStopping(patience=PATIENCE, verbose=False)
        trainer = Trainer(epochs=EPOCHS, learning_rate=LR,
                          callbacks=[early_stop], device=str(DEVICE))
        detector = TransformerOCSVM(
            transformer_model=transformer, trainer=trainer,
            kernel="rbf", nu=OCSVM_NU, gamma="auto",
            n_components=NYSTROEM_COMPONENTS,
            sgd_lr=OCSVM_SGD_LR, sgd_epochs=OCSVM_SGD_EPOCHS)
        return transformer, detector

    if model_type == "pnn":
        input_dim = SEQ_LENGTH * num_features
        model = PNN(input_dim=input_dim, hidden_dim=PNN_HIDDEN_DIM).to(DEVICE)
        return model, None

    if model_type == "prae":
        backbone = BottleneckTransformer(
            num_features=num_features, sequence_length=SEQ_LENGTH, **TRANSFORMER_CFG)
        # num_train_samples is set later per block
        model = PRAE(backbone_model=backbone, num_train_samples=1,
                     lambda_reg=1.0, sigma=PRAE_SIGMA).to(DEVICE)
        return model, None

    raise ValueError(f"Unknown model type: {model_type}")


def train_one_block(model, detector, train_loader, val_loader, model_type):
    """Train the model for one 5-minute block (with early stopping)."""
    if model_type == "transformer_ocsvm":
        # Reset early-stopping state so each day starts fresh
        for cb in detector.trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                cb.reset()
        # Only train the transformer autoencoder here;
        # OC-SVM is fitted once after all days
        detector.trainer.fit(detector.transformer, train_loader, val_loader)
        return

    # PNN / PRAE
    early_stop = EarlyStopping(patience=PATIENCE, verbose=False)
    trainer = Trainer(epochs=EPOCHS, learning_rate=LR,
                      callbacks=[early_stop], device=str(DEVICE))
    trainer.fit(model, train_loader, val_loader)

## Sequential Training Loop

For each of the 24 training days and for each model type:

1. Load and clean the daily file.
2. Compute features.
3. Split the first hour into 5-min train/val blocks.
4. Scale, sequence, and build DataLoaders.
5. Train the model on that block (continuing from the previous day's weights).
6. After all days, save the final model weights and scaler.

In [5]:
MODEL_TYPES = ["transformer_ocsvm", "pnn", "prae"]

# Storage for trained artefacts
trained_models = {}   # model_type -> (model, detector_or_None)
trained_scalers = {}  # model_type -> scaler
feature_name_map = {} # model_type -> list of feature names
training_log = []     # list of dicts for summary

for model_type in MODEL_TYPES:
    logger.info("=" * 80)
    logger.info("MODEL: %s", model_type.upper())
    logger.info("=" * 80)

    model, detector = None, None
    scaler = QuantileScaler()
    feature_names = None
    scaler_fitted = False

    for day_idx in range(NUM_TRAIN_DAYS):
        filepath = FILES[day_idx]
        day_name = os.path.basename(filepath)
        logger.info("Day %d/%d: %s", day_idx + 1, NUM_TRAIN_DAYS, day_name)

        # Load & clean
        df_day = load_and_clean(filepath)

        # Feature engineering
        features = engineer_features(df_day)

        # Re-read df to align after warmup trim
        if len(df_day) > len(features) + WARMUP_STEPS:
            df_day = df_day.iloc[WARMUP_STEPS:].reset_index(drop=True)

        if feature_names is None:
            feature_names = features.columns.tolist()

        # Ensure consistent columns across days
        for col in feature_names:
            if col not in features.columns:
                features[col] = 0.0
        features = features[feature_names]

        # Split first hour
        train_feat, val_feat = split_first_hour_blocks(df_day, features)
        if len(train_feat) < SEQ_LENGTH + 1 or len(val_feat) < SEQ_LENGTH + 1:
            logger.warning("Day %d: insufficient data in first hour, skipping.", day_idx + 1)
            continue

        # Scale & create loaders
        fit = not scaler_fitted
        train_loader, val_loader, scaler, feature_names = scale_and_create_loaders(
            train_feat, val_feat, scaler, model_type, feature_names, fit_scaler=fit)
        scaler_fitted = True

        if train_loader is None:
            logger.warning("Day %d: empty loaders after sequencing, skipping.", day_idx + 1)
            continue

        # Build model on first usable day (or update PRAE sample count)
        num_features = len(feature_names)
        if model is None:
            model, detector = build_fresh_model(model_type, num_features)

        if model_type == "prae":
            n_samples = len(train_loader.dataset)
            model.mu = torch.nn.Parameter(torch.full((n_samples,), 0.5, device=DEVICE))

        # Train
        train_one_block(model, detector, train_loader, val_loader, model_type)

        training_log.append({
            "model": model_type,
            "day": day_idx + 1,
            "file": day_name,
            "train_samples": len(train_loader.dataset),
            "val_samples": len(val_loader.dataset),
        })

    # ----------------------------------------------------------------
    # Fit Nyström OC-SVM once after all days, using the final frozen
    # encoder to re-encode every training day's first-hour data.
    # All latent vectors stay on CUDA -- no numpy round-trip.
    # ----------------------------------------------------------------
    if model_type == "transformer_ocsvm" and detector is not None:
        logger.info("Fitting Nyström OC-SVM on latent representations from all training days...")
        model.eval()
        all_latent = []
        with torch.no_grad():
            for day_idx in range(NUM_TRAIN_DAYS):
                filepath = FILES[day_idx]
                df_day = load_and_clean(filepath)
                feats = engineer_features(df_day)
                if len(df_day) > len(feats) + WARMUP_STEPS:
                    df_day = df_day.iloc[WARMUP_STEPS:].reset_index(drop=True)
                for col in feature_names:
                    if col not in feats.columns:
                        feats[col] = 0.0
                feats = feats[feature_names]
                train_feat, _ = split_first_hour_blocks(df_day, feats)
                if len(train_feat) < SEQ_LENGTH + 1:
                    continue
                scaled = scaler.transform(train_feat.values.astype(np.float32)).astype(np.float32)
                seqs = create_sequences(scaled, SEQ_LENGTH)
                if len(seqs) == 0:
                    continue
                # Encode in batches to avoid OOM -- keep on GPU
                x_tensor = torch.tensor(seqs, dtype=torch.float32)
                ds = TensorDataset(x_tensor, x_tensor)
                loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
                for batch in loader:
                    x = batch[0].to(DEVICE)
                    latent = model.get_representation(x)
                    all_latent.append(latent)
        X_latent = torch.cat(all_latent, dim=0)

        # Set gamma via the median heuristic: γ = 1 / median(‖zᵢ − zⱼ‖²)
        # The previous formula (10 / (d*Var)) produced a gamma that made the
        # RBF kernel ≈ 0 between all distinct points (exp(-20)), collapsing
        # Nyström features to zero and yielding a constant decision function.
        gamma = TransformerOCSVM._median_heuristic_gamma(X_latent)
        detector.ocsvm.set_gamma(gamma)
        logger.info("OC-SVM gamma set to %.6f  (median heuristic, d=%d)", gamma, X_latent.shape[1])

        detector.ocsvm.fit(X_latent)
        logger.info("Nyström OC-SVM fitted on %d latent vectors from %d days.",
                     X_latent.shape[0], NUM_TRAIN_DAYS)

    # Save artefacts
    trained_models[model_type] = (model, detector)
    trained_scalers[model_type] = scaler
    feature_name_map[model_type] = feature_names

    # Persist weights
    weights_path = os.path.join(RESULTS_DIR, f"{model_type}_weights.pth")
    torch.save(model.state_dict(), weights_path)
    logger.info("Saved %s weights to %s", model_type, weights_path)

    # Persist the Nyström OC-SVM (PyTorch module) for the hybrid model
    if model_type == "transformer_ocsvm" and detector is not None:
        detector_path = os.path.join(RESULTS_DIR, f"{model_type}_detector.pth")
        torch.save(detector.ocsvm, detector_path)
        logger.info("Saved Nyström OC-SVM detector to %s", detector_path)

    scaler_path = os.path.join(RESULTS_DIR, f"{model_type}_scaler.pkl")
    joblib.dump(scaler, scaler_path)
    logger.info("Saved scaler to %s", scaler_path)

    # Save feature names
    feat_path = os.path.join(RESULTS_DIR, f"{model_type}_features.txt")
    with open(feat_path, "w") as f:
        f.write("\n".join(feature_names))

logger.info("All models trained.")

2026-02-25 00:10:44 [INFO] training: ================================================================================
2026-02-25 00:10:44 [INFO] training: MODEL: TRANSFORMER_OCSVM
2026-02-25 00:10:44 [INFO] training: ================================================================================
2026-02-25 00:10:44 [INFO] training: Day 1/3: 2015-01-02-TOTF.PA-book.csv.gz
2026-02-25 00:10:47 [INFO] detection.data.preprocessing: Dropped artifact columns: ['Unnamed: 1']
2026-02-25 00:10:47 [INFO] detection.data.preprocessing: Market-hours filter [9.0:00 -- 17.5:00]: kept 563191 / 640429 rows (dropped 77238 pre/post-market).
Epoch 1/1000 - Train Loss: 0.4631 | Val Loss: 0.3597
Epoch 2/1000 - Train Loss: 0.2205 | Val Loss: 0.2672
Epoch 3/1000 - Train Loss: 0.1545 | Val Loss: 0.2317
Epoch 4/1000 - Train Loss: 0.1248 | Val Loss: 0.2146
Epoch 5/1000 - Train Loss: 0.1075 | Val Loss: 0.2043
Epoch 6/1000 - Train Loss: 0.0906 | Val Loss: 0.1975
Epoch 7/1000 - Train Loss: 0.0787 | Val Loss: 0.1917

## Training Summary

In [6]:
log_df = pd.DataFrame(training_log)
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
for mt in MODEL_TYPES:
    subset = log_df[log_df["model"] == mt]
    total = subset["train_samples"].sum() + subset["val_samples"].sum()
    print(f"\n{mt.upper()}")
    print(f"  Days trained: {len(subset)}/{NUM_TRAIN_DAYS}")
    print(f"  Total samples seen: {total:,}")
    model_obj = trained_models[mt][0]
    n_params = sum(p.numel() for p in model_obj.parameters())
    print(f"  Model parameters: {n_params:,}")

print(f"\nSaved artefacts in: {RESULTS_DIR}")
print("Files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f"  {f:40s}  {size/1024:.1f} KB")

log_df


TRAINING SUMMARY

TRANSFORMER_OCSVM
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 1,547,352

PNN
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 141,059

PRAE
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 1,629,309

Saved artefacts in: ..\results
Files:
  pnn_features.txt                          1.9 KB
  pnn_scaler.pkl                            698.8 KB
  pnn_weights.pth                           553.2 KB
  prae_features.txt                         1.9 KB
  prae_scaler.pkl                           698.8 KB
  prae_weights.pth                          7638.2 KB
  transformer_ocsvm_detector.pkl            149801.2 KB
  transformer_ocsvm_detector.pth            505.7 KB
  transformer_ocsvm_features.txt            1.9 KB
  transformer_ocsvm_scaler.pkl              698.8 KB
  transformer_ocsvm_weights.pth             7317.6 KB


,model,day,file,train_samples,val_samples
0,transformer_ocsvm,1,2015-01-02-TOTF.PA-book.csv.gz,38696,50974
1,transformer_ocsvm,2,2015-01-05-TOTF.PA-book.csv.gz,51141,55810
2,transformer_ocsvm,3,2015-01-06-TOTF.PA-book.csv.gz,81957,89420
3,pnn,1,2015-01-02-TOTF.PA-book.csv.gz,38696,50974
4,pnn,2,2015-01-05-TOTF.PA-book.csv.gz,51141,55810
5,pnn,3,2015-01-06-TOTF.PA-book.csv.gz,81957,89420
6,prae,1,2015-01-02-TOTF.PA-book.csv.gz,38696,50974
7,prae,2,2015-01-05-TOTF.PA-book.csv.gz,51141,55810
8,prae,3,2015-01-06-TOTF.PA-book.csv.gz,81957,89420
